In [28]:
import pandas as pd
import re
import numpy as np

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [29]:
df = pd.read_csv("twitter_dataset.csv")

df.head()

,Tweet_ID,Username,Text,Retweets,Likes,Timestamp
0,1,julie81,Party least receive say or single. Prevent pre...,2,25,2023-01-30 11:00:51
1,2,richardhester,Hotel still Congress may member staff. Media d...,35,29,2023-01-02 22:45:58
2,3,williamsjoseph,Nice be her debate industry that year. Film wh...,51,25,2023-01-18 11:25:19
3,4,danielsmary,Laugh explain situation career occur serious. ...,37,18,2023-04-10 22:06:29
4,5,carlwarren,Involve sense former often approach government...,27,80,2023-01-24 07:12:21


In [30]:
stop_words = set(ENGLISH_STOP_WORDS) - {"not", "no", "never"}

In [31]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    clean_words = []
    for word in words:
        if word not in stop_words:
            clean_words.append(word)

    return clean_words

In [32]:
df = df.dropna(subset=["Text"]).copy()

tokens_list = []
for text in df["Text"]:
    tokens_list.append(preprocess(text))

df["tokens"] = tokens_list

In [33]:
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

In [34]:
model = Word2Vec(
    sentences=df["tokens"].tolist(),
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    epochs=20
)

In [35]:
def document_vector(words):
    vectors = []
    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [36]:
tweet_vectors = []
for words in df["tokens"]:
    tweet_vectors.append(document_vector(words))

tweet_vectors = np.array(tweet_vectors)

In [38]:
user_input = input("Enter your search text: ")
user_tokens = preprocess(user_input)
user_vector = document_vector(user_tokens)

Enter your search text:  machine


In [39]:
known_words = False
for word in user_tokens:
    if word in model.wv:
        known_words = True
        break

In [40]:
if known_words == False:
    print("No known words found. Please enter different text.")
else:
    similarities = cosine_similarity(
        user_vector.reshape(1, -1),
        tweet_vectors
    )[0]

In [41]:
top_indices = np.argsort(similarities)[::-1][:5]

In [42]:
print("\nRecommended Tweets:\n")

for i, idx in enumerate(top_indices):
    print("Recommendation", i + 1)
    print("Tweet:", df.iloc[idx]["Text"])
    print("Similarity:", round(similarities[idx], 4))
    print("-" * 50)


Recommended Tweets:

Recommendation 1
Tweet: Government recently later into.
Situation machine kind.
Family everything else actually nearly forget in production. Point here degree present better.
Rate bank loss remain others traditional chance. Office spend election action light author kitchen. Hot them perform perhaps.
Similarity: 0.5491
--------------------------------------------------
Recommendation 2
Tweet: No step leg we blue learn Republican. Table nature go pull machine. Often light everyone long.
Staff think but prevent. When very teacher relationship top maybe physical. Physical detail reflect school special run number letter.
Similarity: 0.5349
--------------------------------------------------
Recommendation 3
Tweet: Set through president camera company without. Like machine trouble significant. Way dinner fast.
Say tonight two send born great. Congress why recent from.
Prove bad fight deep act level kind. Key nice single remember place.
Similarity: 0.5281
----------------